# 07 Verify your dependency
Before you rely on a library (or on your own parser), check it against the source. Draw a sample of holdings, then compare each one with the human-readable schedule in the filing, field by field. Optionally, compare your reconciliation with EdgarTools'.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "params.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from lookthrough import paths, acquire, load, mapping, holdings, clean, reconcile, report, tables, pipeline
P = paths.load_params(ROOT)
RDIR = paths.release_dir(P, ROOT)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
print("release folder:", RDIR.relative_to(ROOT), "| fixture mode:", P["data"]["use_fixture"])

In [ ]:
try:
    panel = tables.load_table(ROOT / P["outputs"]["panel"] / "holdings")
except FileNotFoundError:
    raise SystemExit("Run notebook 06 first: it writes the panel this notebook samples.")
# Up to 10 filings (the 5 largest and up to 5 more at random), 2 holdings each: the largest and one at random.
size = panel.groupby("adsh")["fair_value"].sum().sort_values(ascending=False)
others = size.index[5:].to_series()
chosen = list(size.index[:5]) + list(others.sample(min(5, len(others)), random_state=7))
picks = []
for a in chosen:
    g = panel[panel["adsh"] == a].sort_values("fair_value", ascending=False)
    picks.append(g.head(1))
    if len(g) > 1:
        picks.append(g.iloc[1:].sample(1, random_state=7))
sample = pd.concat(picks)
sub = load.load_sub(RDIR).set_index("adsh")
sample = sample.assign(filing_url=sample["adsh"].map(sub["inlineurl"]),
                       filing_fair_value="", filing_principal="", matches="", note="")
cols = ["adsh", "name", "identifier", "fair_value", "principal", "filing_url", "filing_fair_value", "filing_principal", "matches", "note"]
out = ROOT / P["outputs"]["reports"] / "spot_check.csv"
sample[cols].to_csv(out, index=False)
print(f"{len(sample)} holdings across {sample['adsh'].nunique()} filings -> {out.relative_to(ROOT)}")

**Your turn.** Fill in `spot_check.csv` from the filings: the value in the human-readable schedule, whether it matches, and a note on any difference. Case Study 2B asks for at least 20 holdings across at least 5 filings: on a small panel, add holdings until you reach that. Summarize in `reports/dependency_check.md` (Part 4).

In [ ]:
import importlib.util
if importlib.util.find_spec("edgar") is None or P["data"]["use_fixture"]:
    print("Skipped: needs the pinned edgartools and your real release (fixture BDCs do not exist on EDGAR).")
else:
    from edgar import set_identity
    from edgar.bdc import get_bdc_list
    set_identity(P["sec"]["user_agent"])
    bdcs = get_bdc_list()
    rec = pd.read_csv(ROOT / P["outputs"]["reports"] / "coverage.csv", dtype={"adsh": str, "cik": str})
    for r in rec.head(10).itertuples():
        try:
            inv = bdcs.get_by_cik(int(r.cik)).portfolio_investments()
            print(f"{r.name[:38]:38s} ours {r.gap_pct:+.2%} | edgartools gap {inv.reconciliation_gap:+.2%} "
                  f"(reported total {float(inv.reported_total_fair_value):,.0f})")
        except Exception as e:
            print(f"{r.name[:38]:38s} edgartools comparison failed: {type(e).__name__}: {e}")
    print("Check the period: portfolio_investments() reads the latest filing, which may differ from your release.")